In [1]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('data/scholar-citations.csv', header=None)
df1.columns = ["title", "year", "citations"]

df2 = pd.read_csv('data/scopus-citations.csv', header=None)
df2.columns = ["title", "journal", "date", "citations"]

def stringtoid(s):
    return s.replace(" ", "").replace(".", "").replace(",", "").replace(":", "").lower()

df1["id"] = df1.apply(lambda x: stringtoid(x["title"]), axis=1)
df2["id"] = df2.apply(lambda x: stringtoid(x["title"]), axis=1)

df = df1.merge(df2, how="outer", on=["id"], suffixes=("_scholar", "_scopus")) \
        .sort_values(by=["citations_scopus", "citations_scholar", "year"], ascending=False)

df["isin_scopus"] = df["title_scopus"].notna()
df["isin_scholar"] = df["title_scholar"].notna()
df["title_scholar"].fillna("", inplace=True)
df["title_scopus"].fillna("", inplace=True)
df["title"] = df.apply(lambda x: x["title_scholar"] if len(x["title_scholar"]) > 0 else x["title_scopus"], axis=1)
df["year"] = df.apply(lambda x: x["year"] if x["year"] > 0 else x["date"].split("-")[0], axis=1).astype(int)
df["citations_scholar"] = df["citations_scholar"].fillna(0).astype(int)
df["citations_scopus"] = df["citations_scopus"].fillna(0).astype(int)

In [2]:
df = df[["citations_scopus", "citations_scholar", "isin_scholar", "isin_scopus", "title", "year"]]
df

,citations_scopus,citations_scholar,isin_scholar,isin_scopus,title,year
0,11,20,True,True,Towards a foundational API for resilient distr...,2017
3,8,8,True,True,Towards Conversational OLAP,2020
2,6,9,True,True,A-BI+: a framework for Augmented Business Inte...,2020
1,5,9,True,True,Crop management with the iot: An interdiscipli...,2021
6,5,7,True,True,Augmented Business Intelligence.,2019
7,5,6,True,True,Social BI to understand the debate on vaccines...,2019
5,4,7,True,True,Summarization and visualization of multi-level...,2020
4,3,7,True,True,Towards a conceptual model for data narratives,2020
9,3,4,True,True,The tell-tale cube,2020
12,2,3,True,True,Map-matching on big data: A distributed and ef...,2019


In [11]:
with open('data/summ.txt', 'w') as f:
    f.write(df[["citations_scopus", "citations_scholar"]].sum().to_string().replace("_", ""))

In [3]:
merged = pd.read_csv('data/merge.csv')
diff = merged.merge(df, on=["citations_scholar", "citations_scopus", "title"], how='left', indicator=True)
diff = diff[diff["_merge"] != "both"]
diff

,citations_scopus,citations_scholar,isin_scholar_x,isin_scopus_x,title,year_x,isin_scholar_y,isin_scopus_y,year_y,_merge


In [4]:
diff.to_csv('data/diff.csv', index=False)

In [5]:
df.to_csv('data/merge.csv', index=False)